# Tasks

## P 10.1: Find an image dataset with at least 1000 images. Apply a convolutional neural network to classify the images and calculate all evaluation metrics. The quality of your solution and the accuracy your model produces will affect your mark. So, try to get the best accuracy possible (10%).

## NOTE: You should comment on your code and explain what each part of the code does.

In [2]:
############# WRITE YOUR CODE IN THIS CELL (IF APPLICABLE)  ####################
# Import necessary libraries
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix

# 1. Data Preparation and Transformation
# normalize the images and convert to tensors
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load the CIFAR10 training & test datasets
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 2. Define the CNN Architecture
class CNNClassifier(nn.Module):
    def __init__(self):
        super(CNNClassifier, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 8 * 8, 256)
        self.fc2   = nn.Linear(256, 10)
        self.relu  = nn.ReLU()
        self.drop  = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64*8*8)
        x = self.drop(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# 3. Instantiate model, loss function, and optimizer
model = CNNClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Training Loop
epochs = 15
for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

print("\n🌟 Training Finished!")

# 5. Evaluate the Model
model.eval()
all_labels = []
all_preds  = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_labels += labels.tolist()
        all_preds  += preds.tolist()

# Classification Report & Confusion Matrix
print("\n📊 Evaluation Metrics")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))
print("\nConfusion Matrix:\n", confusion_matrix(all_labels, all_preds))




ModuleNotFoundError: No module named 'torch'

############ WRITE YOUR REPORT IN THIS CELL (IF APPLICABLE) #############

## **CIFAR-10 Image Classification Using a Convolutional Neural Network**
### 1. Introduction

This task involves building and evaluating a **Convolutional Neural Network (CNN)** to perform **multi-class image classification** on the CIFAR-10 dataset. CNNs are widely used in computer vision because they can automatically learn spatial hierarchies of features from raw image data. The objective of this experiment is to design a CNN, train it effectively, and evaluate its performance using standard classification metrics.

### 2. Dataset and Preprocessing

The **CIFAR-10 dataset** consists of **60,000 RGB images** with a resolution of **32×32 pixels**, divided into **50,000 training images** and **10,000 test images** across **10 object categories** (such as airplane, automobile, bird, and ship).

Before training, all images were converted to tensors and normalized using a mean and standard deviation of **0.5 for each color channel**. Normalization ensures that pixel values are centered and scaled, which improves training stability and convergence speed. The dataset was loaded using PyTorch utilities and split into mini-batches of **64 samples** using DataLoader.

### 3. CNN Architecture

The CNN model was designed with the following structure:

* Two **convolutional layers** with 32 and 64 filters respectively, each using a 3×3 kernel and padding
* **ReLU activation functions** to introduce non-linearity
* **Max-pooling layers** to reduce spatial dimensions and computational cost
* A **fully connected layer** with 256 neurons for high-level feature learning
* A **dropout layer (25%)** to reduce overfitting
* A final **output layer** with 10 neurons corresponding to the CIFAR-10 classes

This architecture allows the model to progressively learn low-level and high-level visual features.

### 4. Training Process

The model was trained for **15 epochs** using the **Adam optimizer** with a learning rate of **0.001**. The **cross-entropy loss function** was used, as it is appropriate for multi-class classification tasks.
During training, the loss consistently decreased across epochs, indicating that the model was learning meaningful patterns from the data.

### 5. Model Evaluation

After training, the model was evaluated using the test dataset. Performance was measured using:

* **Precision, recall, and F1-score** for each class
* An overall **classification report**
* A **confusion matrix** to analyze class-wise prediction performance

The results show that the CNN achieved reasonable classification accuracy across most classes. Some misclassifications occurred between visually similar classes, which is common in compact datasets like CIFAR-10.

### 6. Conclusion

This experiment demonstrates that a properly designed **Convolutional Neural Network** can effectively classify images in the CIFAR-10 dataset. The use of convolutional layers, pooling, dropout, and the Adam optimizer contributed to stable training and good generalization. Further improvements could be achieved by increasing network depth, applying data augmentation, or tuning hyperparameters.
